<a href="https://colab.research.google.com/github/CharlestonA/IIITM_Codes/blob/main/FCNN_Classification_Keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml

# --- SECTION 2: DATA PREPARATION AND PROBLEM SETUP ---

# 1. Dataset Selection & Problem Definition
# Problem: Binary Classification (Predict if income >50K)
print("Loading Adult Income dataset...")
adult = fetch_openml(data_id=1590, as_frame=True, parser='auto')
df = adult.frame

# 2. Dataset Exploration
print("\n--- Initial Data Exploration ---")
print(f"Dataset Shape: {df.shape}")
print("\nData Types:\n", df.dtypes)
print("\nBasic Statistics:\n", df.describe())
print("\nClass Balance:\n", df['class'].value_counts(normalize=True))

# 3. Handling Missing Values & Feature Selection
# Identifying and dropping rows with missing values
df = df.dropna() #
print(f"\nShape after removing missing values: {df.shape}")

# Identifying Input Features (X) and Target Variable (y)
X = df.drop(columns=['class'])
y = df['class']

# 4. Preprocessing Steps for Neural Networks
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['category', 'object']).columns

# Encoding and Scaling
# Scaling is crucial for stable and faster NN training
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features), # Feature scaling
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # Categorical encoding
    ])

# 5. Target Label Encoding
# Binary classification requires numerical labels (0 or 1)
y_encoded = y.apply(lambda x: 1 if '>' in str(x) else 0).astype('float32')

In [ ]:
# 6. Data Splitting
# Role of validation data: Tuning and monitoring overfitting
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

# 7. Transformation and Reshaping
X_train = preprocessor.fit_transform(X_train).toarray().astype('float32')
X_val = preprocessor.transform(X_val).toarray().astype('float32')
X_test = preprocessor.transform(X_test).toarray().astype('float32')

# 8. Basic Sanity Checks
print("\n--- Post-Processing Sanity Checks ---")
print(f"Training Input Shape: {X_train.shape}") # Shape consistency
print(f"Validation Target Shape: {y_val.shape}")
print(f"Mean of scaled numeric features (should be approx 0): {X_train[:,0].mean():.4f}") # Distribution check
print(f"Max value in scaled data: {X_train.max():.4f}")

# --- SECTION 3: BUILDING AND TRAINING FCFNN MODELS ---

import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = X_train.shape[1] # Based on number of features

model = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(input_shape,)), # Hidden layer 1
    layers.Dense(32, activation='relu'), # Hidden layer 2
    layers.Dropout(0.2), # Regularization to prevent overfitting
    layers.Dense(1, activation='sigmoid') # Output layer for binary classification
])

model.compile(
    optimizer='adam', # Optimization approach
    loss='binary_crossentropy', # Suitable loss function
    metrics=['accuracy'] # Relevant evaluation metric
)

print("\n--- Starting Model Training ---")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val), # Validation role: Monitoring
    epochs=20,
    batch_size=64, # Training parameters
    verbose=1
)

# --- SECTION 5 & 6: EVALUATION AND VISUAL ANALYSIS ---
# Evaluate on test set
results = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {results[1]:.4f}")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss') # Monitoring overfitting
plt.title('Loss Curves (Cross-Entropy)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Curves')
plt.legend()
plt.show()